# BAS-HAR: fine-tune YOLO11n on the demo-table dataset (free GPU)

Follow `docs/finetune-runbook.md`, step 4.

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. In cell 2, set `ULTRALYTICS_VERSION` to the version on your laptop
   (`python -c "import ultralytics; print(ultralytics.__version__)"` inside the project `.venv`).
   Training with the same version guarantees the weights load in your backend.
3. **Runtime → Run all.** Cell 3 asks you to upload `dataset/bas_reviewed_colab.zip`.
4. The last cell downloads `bas_custom_results.zip` (the weights + training curves).

In [ ]:
# 1. Check that a GPU is attached (should print a Tesla T4 or similar)
!nvidia-smi -L || echo "NO GPU - Runtime > Change runtime type > T4 GPU, then run again"

In [ ]:
# 2. Same Ultralytics version as the laptop
ULTRALYTICS_VERSION = ""   # <-- EDIT THIS: the exact version printed on your laptop
assert ULTRALYTICS_VERSION, "Set ULTRALYTICS_VERSION to your laptop's ultralytics version first"
%pip install -q "ultralytics=={ULTRALYTICS_VERSION}"
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())

In [ ]:
# 3. Upload dataset/bas_reviewed_colab.zip (written by tools/import_reviewed.py)
import os, zipfile
from google.colab import files
up = files.upload()
name = next(iter(up))
os.makedirs("/content/BAS-HAR", exist_ok=True)
zipfile.ZipFile(name).extractall("/content/BAS-HAR")
%cd /content/BAS-HAR
!cat dataset/bas_reviewed/split.json
!echo train images: $(ls dataset/bas_reviewed/images/train | wc -l)  val images: $(ls dataset/bas_reviewed/images/val | wc -l)

In [ ]:
# 4. Train (same script and augmentations as on the laptop). ~10-20 min on a T4 for ~600 images.
!python tools/train.py --data dataset/bas_reviewed/data.yaml --epochs 60 --batch 16 --imgsz 640 --device 0

In [ ]:
# 5. Download the weights + training curves
!rm -f bas_custom_results.zip && zip -qr bas_custom_results.zip backend/models/bas_custom.pt runs/bas -x "runs/bas/weights/last.pt"
files.download("bas_custom_results.zip")